# Análisis Exploratorio de Datos (EDA) Inicial - ODEPA

Este notebook contiene la estructura inicial para el diagnóstico y exploración del dataset de precios hortofrutícolas de la Oficina de Estudios y Políticas Agrarias (ODEPA), integrando los datos de los años 2025 y 2026.

> **Nota Importante:** En esta primera fase, **no se aplica ningún tipo de normalización de datos ni escalado de variables**. El objetivo principal es realizar un diagnóstico empírico de la calidad del dataset, comprender su estructura básica, analizar la cardinalidad de sus variables categóricas en ambos años, rastrear registros con valores en cero y extraer componentes temporales básicos.

## Configuración de la Terminal (Fish Shell)

Si estás utilizando la terminal **Fish Shell** en Linux, puedes configurar tu entorno de la siguiente manera:

1. **Crear entorno virtual:**
   ```fish
   python3 -m venv .venv
   ```
2. **Activar en Fish:**
   ```fish
   source .venv/bin/activate.fish
   ```
3. **Instalar dependencias:**
   ```fish
   pip install -r requirements.txt
   ```
4. **Registrar kernel de Jupyter:**
   ```fish
   python3 -m ipykernel install --user --name=prediccion-precios --display-name "Predicción Precios (ODEPA)"
   ```
5. **Ejecutar Jupyter:**
   ```fish
   jupyter notebook
   ```

## 1. Carga de los Datasets (2025 y 2026)

En esta celda importamos las librerías necesarias (`pandas`, `numpy`, `matplotlib`, `seaborn`) y realizamos la lectura inicial de los archivos de datos de 2025 y 2026 para comparar sus estructuras.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas a los datasets
csv_2025_path = 'dataset/2025.csv'
csv_2026_path = 'dataset/2026.csv'

# Carga básica
print("Cargando archivos...")
df25 = pd.read_csv(csv_2025_path)
df26 = pd.read_csv(csv_2026_path)
print(f"Carga completa. Registros 2025: {df25.shape[0]} | Registros 2026: {df26.shape[0]}")

## 2. Diagnóstico Estructural Comparativo

Evaluamos las dimensiones de ambos conjuntos de datos, validamos que tengan exactamente el mismo esquema de columnas y verificamos que los tipos de datos sean homogéneos.

In [ ]:
# 1. Verificar coincidencia de columnas
diff_cols = set(df25.columns) ^ set(df26.columns)
print("Diferencia en las columnas de los datasets:", diff_cols if diff_cols else "Ninguna (Esquemas idénticos)")

# 2. Tipos de datos
print("\n--- Tipos de datos en 2026 ---")
print(df26.dtypes)

# 3. Primeras filas
print("\n--- Muestra del dataset 2026 ---")
df26.head()

## 3. Análisis de Ceros (Valores Vacíos o Nulos Implícitos)

Buscamos registros con Volumen o Precio promedio igual a cero o con formato de texto `'0,0000'` en ambos conjuntos de datos. Esto nos permite comprobar la consistencia de la calidad del dato entre años.

In [ ]:
def analizar_ceros(df, year_label):
    vol_zero_mask = (
        (df['Volumen'] == 0) | 
        (df['Volumen'] == 0.0) | 
        (df['Volumen'].astype(str).str.strip() == '0') | 
        (df['Volumen'].astype(str).str.strip() == '0,0000')
    )
    
    precio_zero_mask = (
        (df['Precio promedio'] == 0) | 
        (df['Precio promedio'] == 0.0) | 
        (df['Precio promedio'].astype(str).str.strip() == '0') | 
        (df['Precio promedio'].astype(str).str.strip() == '0,0000')
    )
    
    print(f"--- Análisis de ceros para {year_label} ---")
    print(f"Volumen en cero: {vol_zero_mask.sum()}")
    print(f"Precio promedio en cero: {precio_zero_mask.sum()}")
    print(f"Registros afectados (al menos uno en cero): {(vol_zero_mask | precio_zero_mask).sum()}\n")

analizar_ceros(df25, "2025")
analizar_ceros(df26, "2026")

## 4. Análisis de Cardinalidad y Nuevas Categorías (2025 vs. 2026)

Es vital analizar si en el año 2026 se introducen nuevas categorías (nuevos mercados, productos, orígenes o tipos de envases) que no estaban presentes en 2025. Esto justifica y define cómo manejaremos las variables categóricas (por ejemplo, el uso de Target Encoding con suavizado o valores por defecto para clases desconocidas).

In [ ]:
categorical_cols = ['Calidad', 'Subsector', 'Mercado', 'Producto', 'Variedad / Tipo', 'Origen', 'Unidad de comercializacion']

print(f"{'Columna':<28} | {'Únicos 2025':<12} | {'Únicos 2026':<12} | {'Nuevas Categorías en 2026'}")
print("-" * 85)

for col in categorical_cols:
    set25 = set(df25[col].dropna().unique())
    set26 = set(df26[col].dropna().unique())
    nuevos = set26 - set25
    
    print(f"{col:<28} | {len(set25):<12} | {len(set26):<12} | {len(nuevos)}")
    if len(nuevos) > 0:
        print(f"   -> Listado de nuevas: {list(nuevos)[:10]}")

print("-" * 85)

## 5. Análisis Temporal y de Estaciones

Convertimos las fechas y extraemos el mes, el día de la semana y la estación del año correspondiente al Hemisferio Sur (Chile). Evaluamos si los datos de 2026 siguen cubriendo todas las estaciones de forma representativa.

In [ ]:
def procesar_tiempo(df, label):
    df_t = df.copy()
    df_t['Fecha'] = pd.to_datetime(df_t['Fecha'])
    df_t['Mes'] = df_t['Fecha'].dt.month
    
    # Clasificación estacional en Chile (Hemisferio Sur)
    def get_season_chile(month):
        if month in [12, 1, 2]:
            return 'Verano'
        elif month in [3, 4, 5]:
            return 'Otoño'
        elif month in [6, 7, 8]:
            return 'Invierno'
        elif month in [9, 10, 11]:
            return 'Primavera'
        return 'Desconocido'
    
    df_t['Estacion'] = df_t['Mes'].apply(get_season_chile)
    print(f"--- Distribución estacional para {label} ---")
    print(df_t['Estacion'].value_counts())
    print("\n")
    return df_t

df25_temp = procesar_tiempo(df25, "2025")
df26_temp = procesar_tiempo(df26, "2026")